# 🎲 Chapter 3 — VC-dimension & fitting pure noise (hands-on)

Companion to Chapter 3 of *Foundations of Machine Learning*. You'll run the **Rademacher
experiment** — fit a model to randomly shuffled labels and watch capacity reveal itself —
then work out shattering and growth functions yourself.

**Learn** (visible checks) → **Apply** (hidden auto-graded, hints) → **Appendix** (locked solutions).


In [ ]:
# Setup — run me first
try:
    import sklearn  # noqa
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable,'-m','pip','install','-q','scikit-learn'])
import base64, marshal, itertools, random
def check(name, fn, hint=''):
    try: ok=bool(fn())
    except Exception as e: ok=False; hint=(hint+f'  (raised {type(e).__name__}: {e})').strip()
    print(f"{'✅' if ok else '❌'}  {name}" + (f'  — {hint}' if not ok and hint else '')); return ok
def _eq(a,b,tol=1e-6):
    if isinstance(b,(list,tuple)): return isinstance(a,(list,tuple)) and len(a)==len(b) and all(_eq(x,y,tol) for x,y in zip(a,b))
    if isinstance(b,bool): return a is b or a==b
    if isinstance(b,float):
        try: return abs(a-b)<=tol
        except Exception: return False
    return a==b
_TESTS={"p1": "WwMAAAApAqkB6QEAAABbAgAAACkB6QAAAAApAXIBAAAAKQKpAekCAAAAWwQAAAApAnICAAAAcgIAAAApAnICAAAAcgEAAAApAnIBAAAAcgIAAAApAnIBAAAAcgEAAAApAqkB6QMAAABbCAAAACkDcgIAAAByAgAAAHICAAAAKQNyAgAAAHICAAAAcgEAAAApA3ICAAAAcgEAAAByAgAAACkDcgIAAAByAQAAAHIBAAAAKQNyAQAAAHICAAAAcgIAAAApA3IBAAAAcgIAAAByAQAAACkDcgEAAAByAQAAAHICAAAAKQNyAQAAAHIBAAAAcgEAAAA=", "p2": "WwQAAAApAqkCWwMAAADpAQAAAOkCAAAA6QMAAABbAwAAAOkAAAAAcgQAAAByAQAAAFQpAqkCWwMAAAByAQAAAHICAAAAcgMAAABbAwAAAHIBAAAAcgQAAAByBAAAAEYpAqkCWwMAAAByAwAAAHIBAAAAcgIAAABbAwAAAHIBAAAAcgQAAAByBAAAAFQpAqkCWwIAAADpBQAAAHIBAAAAWwIAAAByAQAAAHIEAAAAVA==", "p3": "WwMAAAApAqkB6QEAAADpAgAAACkCqQHpAwAAAOkHAAAAKQKpAekFAAAA6RAAAAA="}
_HINTS={"p1": ["itertools.product((0,1), repeat=n) gives every combination.", "Return a list of tuples, in binary-counting order."], "p2": ["A threshold rule h(x)=1 if x>t labels everything above t as 1.", "So after sorting by x, the labels must be all 0s followed by all 1s \u2014 never a 0 after a 1."], "p3": ["An interval is fixed by choosing 2 cut points among the m+1 gaps between/around sorted points.", "That's C(m+1,2) = m(m+1)/2, plus 1 for the empty (all-zero) labelling."]}
_SOLUTIONS={}
def grade(pid, fn):
    cs=marshal.loads(base64.b64decode(_TESTS[pid]))
    for k,(args,exp) in enumerate(cs,1):
        try: got=fn(*args)
        except Exception as e:
            print(f"❌ {pid}: hidden case {k}/{len(cs)} raised {type(e).__name__}: {e}. Try hint('{pid}')."); return False
        if not _eq(got,exp):
            print(f"❌ {pid}: hidden case {k}/{len(cs)} failed — not right yet. Try hint('{pid}')."); return False
    print(f'✅ {pid}: all {len(cs)} hidden tests passed 🎉'); return True
def hint(pid,n=None):
    hs=_HINTS.get(pid,[]); rng=range(len(hs)) if n is None else [n-1]
    [print(f'Hint {i+1}: {hs[i]}') for i in rng]
def reveal(pid):
    if pid not in _SOLUTIONS: print('🔒 Locked — run the Appendix cell at the bottom first.'); return
    print(base64.b64decode(_SOLUTIONS[pid]).decode())
print('Toolkit ready ✓')


## Part A — Learn: how well can your model fit *nonsense*?
The Rademacher idea in one experiment. Take real features, **throw the labels away and replace
them with coin flips**, then fit. A model that scores well on noise has enough capacity to
memorize — so its score on the *real* labels proves very little.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

X, y_real = make_classification(n_samples=200, n_features=8, random_state=0)
rng = random.Random(0)
y_noise = [rng.randint(0, 1) for _ in range(len(y_real))]    # pure coin flips

def fit_score(model, y):
    return model.fit(X, y).score(X, y)      # TRAINING accuracy on purpose

deep_tree  = lambda: DecisionTreeClassifier(random_state=0)              # huge capacity
linear     = lambda: LogisticRegression(max_iter=1000)                   # small capacity

print('fitting RANDOM labels (pure noise):')
print(f'  deep tree : {fit_score(deep_tree(), y_noise):.3f}')
print(f'  linear    : {fit_score(linear(), y_noise):.3f}')
print('\nfitting the REAL labels:')
print(f'  deep tree : {fit_score(deep_tree(), y_real):.3f}')
print(f'  linear    : {fit_score(linear(), y_real):.3f}')

check('the deep tree fits pure noise almost perfectly (capacity to memorize)',
      lambda: fit_score(deep_tree(), y_noise) > 0.95)
check('the linear model canNOT fit noise well (limited capacity)',
      lambda: fit_score(linear(), y_noise) < 0.85)
check('so a perfect training score from the tree is NOT evidence of learning',
      lambda: fit_score(deep_tree(), y_real) > 0.95 and fit_score(deep_tree(), y_noise) > 0.95)


### What you just measured
The tree scored ~100% on **coin flips**. That is the empirical Rademacher signal: this class can
realise essentially any labelling of your sample, so its perfect score on the real labels tells
you nothing about generalization. The linear model, unable to fit noise, earns its score.

## Part B — Apply  🔒
Implement each, run `grade('pN', fn)` (hidden tests). `hint('pN')` if stuck.

### p1 · `all_labelings(n)`
Return **every** labelling of `n` points as a list of 0/1 tuples, in binary-counting order (so `n=1` → `[(0,), (1,)]`). This is the set shattering must cover — all 2ⁿ of them.

In [ ]:
import itertools
def all_labelings(n):
    # TODO: all 2**n tuples of 0/1
    pass

grade('p1', all_labelings)
# hint('p1')


### p2 · `realizable_by_threshold(xs, labels)`
A threshold rule labels 1 exactly when `x > t`. Return `True` if some `t` produces `labels` for points `xs`. (This is why a threshold shatters 1 point but never 2.)

In [ ]:
def realizable_by_threshold(xs, labels):
    # TODO: after sorting by x, labels must be 0...0 then 1...1
    pass

grade('p2', realizable_by_threshold)
# hint('p2')


### p3 · `growth_intervals(m)`
How many distinct labellings of `m` points can an **interval** rule (1 inside `[a,b]`) produce? Return the count — the growth function.

In [ ]:
def growth_intervals(m):
    # TODO: count labellings an interval can make on m points
    pass

grade('p3', growth_intervals)
# hint('p3')


### ✅ Score

In [ ]:
import io, contextlib
_pairs=[('p1', all_labelings), ('p2', realizable_by_threshold), ('p3', growth_intervals)]
done=0
for pid, fn in _pairs:
    buf=io.StringIO()
    with contextlib.redirect_stdout(buf): done+=grade(pid, fn)
print(f'\nApply score: {done} / 3' + ('  — capacity theory mastered 🏆' if done==3 else ''))


---
## Appendix — Solutions (locked 🔒)
Try first, then run this and call `reveal('p1')` etc.

In [ ]:
_SOLUTIONS={"p1": "aW1wb3J0IGl0ZXJ0b29scwpkZWYgYWxsX2xhYmVsaW5ncyhuKToKICAgIHJldHVybiBbdHVwbGUodCkgZm9yIHQgaW4gaXRlcnRvb2xzLnByb2R1Y3QoKDAsMSksIHJlcGVhdD1uKV0=", "p2": "ZGVmIHJlYWxpemFibGVfYnlfdGhyZXNob2xkKHhzLCBsYWJlbHMpOgogICAgIyBzb3J0IGJ5IHg7IGEgdGhyZXNob2xkIHJ1bGUgY2FuIG9ubHkgcHJvZHVjZSAwLi4uMCB0aGVuIDEuLi4xCiAgICBzZWVuX29uZSA9IEZhbHNlCiAgICBmb3IgXywgbGFiIGluIHNvcnRlZCh6aXAoeHMsIGxhYmVscykpOgogICAgICAgIGlmIGxhYiA9PSAxOiBzZWVuX29uZSA9IFRydWUKICAgICAgICBlbGlmIHNlZW5fb25lOiByZXR1cm4gRmFsc2UKICAgIHJldHVybiBUcnVl", "p3": "ZGVmIGdyb3d0aF9pbnRlcnZhbHMobSk6CiAgICAjIGNob29zZSAyIGN1dCBwb2ludHMgZnJvbSBtKzEgZ2FwcywgcGx1cyB0aGUgYWxsLXplcm8gbGFiZWxsaW5nCiAgICByZXR1cm4gbSoobSsxKS8vMiArIDE="}
print('🔓 Unlocked. reveal("p1") / reveal("p2") / reveal("p3").')
